# Tempered participant weighting: leave-one-source-out

A theory-guided compromise after notebook 23: each source-by-label cell remains balanced, while a participant's contribution is weighted by `windows_per_person^-0.5`. This reduces recording-length dominance without discarding the information carried by longer walking records. No non-primary or frozen data are read.

In [1]:
from pathlib import Path
import sys,gc
import numpy as np,pandas as pd,torch
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.metrics import roc_auc_score,balanced_accuracy_score,brier_score_loss
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P=ROOT/'data'/'processed'; D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',D)
x=np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'),np.load(P/'sint_maartenskliniek_external_windows_float32.npy')]); m=pd.concat([pd.read_csv(P/'validated_window_metadata.csv'),pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')],ignore_index=True); m=m[m.label.isin(['healthy','stroke'])].reset_index(drop=True); m['y']=m.label.eq('stroke').astype(int); m['source']=m.dataset_id; m['group']=m.participant_key.astype(str)
def weights(frame,alpha):
    # alpha=0 reproduces source/class window weighting; alpha=1 gives equal participant mass.
    pc=frame.groupby('group').size(); raw=frame.group.map(lambda g: pc[g]**(-alpha)).to_numpy(float); keys=pd.MultiIndex.from_frame(frame[['source','y']]); totals=pd.Series(raw).groupby([frame.source.to_numpy(),frame.y.to_numpy()]).sum(); den=np.asarray(keys.map(totals),float); return torch.tensor(raw/den,dtype=torch.double)
def metrics(net,arr,meta,mean,std):
    with torch.inference_mode(): p=torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
    g=meta.assign(p=p).groupby(['group','y'],as_index=False).p.mean(); return {'participants':len(g),'healthy':int((g.y==0).sum()),'stroke':int((g.y==1).sum()),'auroc':roc_auc_score(g.y,g.p),'balanced_accuracy':balanced_accuracy_score(g.y,g.p>=.5),'healthy_specificity':float((g.loc[g.y==0,'p']<.5).mean()),'brier':brier_score_loss(g.y,g.p)}
rows=[]
for held in sorted(m.source.unique()):
    tr=m.source.ne(held).to_numpy(); va=~tr
    for seed in [42,137,202]:
        for name,alpha in [('legacy_window_alpha_0',0.),('tempered_alpha_0p5',.5)]:
            torch.manual_seed(seed); tx=x[tr]; mean,std=tx.reshape(-1,3).mean(0),tx.reshape(-1,3).std(0).clip(1e-4); z=torch.from_numpy(((tx-mean)/std).transpose(0,2,1).astype('float32')); y=torch.from_numpy(m.loc[tr,'y'].to_numpy('float32')); dl=DataLoader(TensorDataset(z,y),128,sampler=WeightedRandomSampler(weights(m.loc[tr],alpha),len(z),replacement=True,generator=torch.Generator().manual_seed(seed+1000)))
            net=StrokeGaitInception().to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
            for _ in range(8):
                net.train()
                for a,b in dl: opt.zero_grad(); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)),b.to(D)); loss.backward(); opt.step()
            net.eval(); rows.append({'held_out_source':held,'seed':seed,'mode':name,'alpha':alpha,**metrics(net,x[va],m.loc[va],mean,std)}); del net,opt,dl,z,y; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',held,seed,name)
out=pd.DataFrame(rows); out.to_csv(P/'tempered_sampler_leave_one_source_out.csv',index=False); print(out.groupby(['held_out_source','mode'])[['auroc','balanced_accuracy','healthy_specificity','brier']].agg(['mean','std']).round(4))

device: cuda


complete felius_2024 42 legacy_window_alpha_0


complete felius_2024 42 tempered_alpha_0p5


complete felius_2024 137 legacy_window_alpha_0


complete felius_2024 137 tempered_alpha_0p5


...

complete voisard_2025 42 tempered_alpha_0p5


complete voisard_2025 137 legacy_window_alpha_0


complete voisard_2025 137 tempered_alpha_0p5


complete voisard_2025 202 legacy_window_alpha_0


complete voisard_2025 202 tempered_alpha_0p5
                                             auroc         balanced_accuracy  \
                                              mean     std              mean   
held_out_source      mode                                                      
felius_2024          legacy_window_alpha_0  0.8792  0.0058            0.7582   
                     tempered_alpha_0p5     0.8820  0.0049            0.7813   
sint_maartenskliniek legacy_window_alpha_0  0.8900  0.0218            0.8417   
                     tempered_alpha_0p5     0.8783  0.0126            0.8417   
voisard_2025         legacy_window_alpha_0  0.9303  0.0296            0.8518   
                     tempered_alpha_0p5     0.9070  0.0314            0.8042   

                                                   healthy_specificity  \
                                               std                mean   
held_out_source      mode                                                
felius_2024 

Tempered weighting advances only if it is non-inferior to the legacy sampler on every held-out source and improves at least one clinically important transport metric without worsening calibration. Otherwise the legacy sampler remains the current three-channel reference and sampling is not the main remaining problem.